# Analisis Exploratorio de Datos
## Alquileres de Bienes Inmuebles en Ecuador
---
**Laboratorio de Ciencia de Datos — Proceso de Seleccion Tecnico de Investigacion**
**Escuela Politecnica Nacional — Laboratorio ADA**

Este notebook cubre el **Paso 1**:
- Carga y limpieza del dataset
- Normalizacion de la columna `Lugar`
- Manejo de valores faltantes
- Analisis descriptivo completo
- Creacion de la columna `Tipo de Precio por Lugar`


In [ ]:
!pip install pandas numpy matplotlib seaborn plotly missingno --quiet

In [ ]:
import re, warnings, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import missingno as msno
import plotly.express as px
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", "{:,.2f}".format)

COLOR_MAIN = "#1a5276"
COLOR_ACC  = "#2e86c1"
COLOR_WARN = "#e74c3c"
os.makedirs("figs", exist_ok=True)
print("Librerias cargadas correctamente")


## 1. Carga del Dataset

In [ ]:
# Carga robusta con manejo de multiples encodings
# El CSV puede presentar artefactos latin-1/UTF-8 (Ã± -> n~)

DATA_PATH = "data/alquileres_ecuador.csv"   # ajustar si es necesario

def load_csv_robust(path: str) -> pd.DataFrame:
    encodings = ["utf-8", "latin-1", "cp1252", "utf-8-sig"]
    for enc in encodings:
        try:
            df = pd.read_csv(path, encoding=enc)
            # Reparar artefactos de doble-codificacion
            for col in df.select_dtypes("object").columns:
                try:
                    df[col] = (
                        df[col]
                        .str.encode("latin-1", errors="ignore")
                        .str.decode("utf-8", errors="ignore")
                    )
                except Exception:
                    pass
            print(f"Archivo cargado con encoding '{enc}'")
            return df
        except Exception as e:
            print(f"  encoding '{enc}' fallo: {e}")
    raise ValueError("No se pudo cargar el archivo.")

df_raw = load_csv_robust(DATA_PATH)

# Estandarizar nombres de columnas
df_raw.columns = (
    df_raw.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9_]", "_", regex=True)
    .str.replace(r"_+", "_", regex=True)
    .str.strip("_")
)

COLUMN_MAP = {
    "titulo"          : "titulo",
    "precio"          : "precio",
    "provincia"       : "provincia",
    "lugar"           : "lugar_raw",
    "num__dormitorios": "num_dormitorios",
    "num__banos"      : "num_banos",
    "area"            : "area",
    "num__garages"    : "num_garages",
}
df_raw = df_raw.rename(columns={c: COLUMN_MAP.get(c, c) for c in df_raw.columns})
print(f"Shape del dataset: {df_raw.shape}")
df_raw.head(3)


In [ ]:
df_raw.info()
df_raw.head()

## 2. Limpieza y Normalizacion

### 2.1 Normalizacion de la columna Lugar

In [ ]:
# Formato observado: "Provincia, Sector, Ciudad, Ecuador"
#                 o: "Provincia, Ciudad, Ecuador"
# Objetivo: extraer ciudad y sector de forma robusta.

STOPWORDS = {"ecuador", ""}

def normalizar_lugar(row: pd.Series) -> pd.Series:
    raw  = str(row.get("lugar_raw", "")).strip()
    prov = str(row.get("provincia", "")).strip().lower()
    partes = [p.strip() for p in raw.split(",")]
    filtradas = [
        p for p in partes
        if p.lower() not in STOPWORDS and p.lower() != prov
    ]
    ciudad = filtradas[-1] if filtradas else "Desconocido"
    sector = filtradas[-2] if len(filtradas) >= 2 else ciudad
    ciudad = re.sub(r"\s+", " ", ciudad).title().strip()
    sector = re.sub(r"\s+", " ", sector).title().strip()
    return pd.Series({"ciudad": ciudad, "sector": sector})

ubicacion = df_raw.apply(normalizar_lugar, axis=1)
df = df_raw.join(ubicacion).drop(columns=["lugar_raw"], errors="ignore")
df["lugar"] = df["ciudad"]   # columna canonica para el modelo

print("Ciudades unicas detectadas (top 20):")
print(df["ciudad"].value_counts().head(20))


### 2.2 Limpieza numerica y manejo de valores faltantes

In [ ]:
NUM_COLS = ["precio", "num_dormitorios", "num_banos", "area", "num_garages"]

for col in NUM_COLS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Eliminar outliers severos en precio
precio_max = df["precio"].quantile(0.995)
precio_min = 10
mask = (df["precio"] >= precio_min) & (df["precio"] <= precio_max)
n_elim = (~mask).sum()
df = df[mask].copy()
print(f"Filas eliminadas por precio fuera de rango: {n_elim}")

print("\nValores faltantes por columna:")
faltantes = df.isnull().sum().rename("NaN").to_frame()
faltantes["%"] = (faltantes["NaN"] / len(df) * 100).round(2)
print(faltantes[faltantes["NaN"] > 0])


In [ ]:
# Imputacion contextual por provincia+ciudad -> mediana
for col in ["num_dormitorios", "num_banos", "area", "num_garages"]:
    med_grupo  = df.groupby(["provincia", "ciudad"])[col].transform("median")
    med_global = df[col].median()
    df[col] = df[col].fillna(med_grupo).fillna(med_global)

for col in ["provincia", "ciudad", "sector", "lugar"]:
    if col in df.columns:
        df[col] = df[col].fillna("Desconocido")

df = df.dropna(subset=["precio"])
print(f"Dataset limpio: {df.shape[0]:,} registros, {df.shape[1]} columnas")


In [ ]:
# Mapa de valores faltantes
fig, ax = plt.subplots(figsize=(10, 4))
msno.matrix(df_raw[NUM_COLS + ["provincia"]], ax=ax,
            color=(0.1, 0.32, 0.47), fontsize=11, sparkline=False)
ax.set_title("Mapa de valores faltantes - Dataset original", fontsize=13, pad=12)
plt.tight_layout()
plt.savefig("figs/missing_values.png", dpi=150, bbox_inches="tight")
plt.show()


## 3. Analisis Descriptivo

### 3.1 Totales generales

In [ ]:
print("=" * 60)
print("RESUMEN GENERAL")
print("=" * 60)
print(f"  Total de propiedades (limpias)  : {len(df):,}")
print(f"  Provincias unicas               : {df['provincia'].nunique()}")
print(f"  Ciudades unicas                 : {df['ciudad'].nunique()}")
print(f"  Rango de precios (USD)          : [{df['precio'].min():,.0f} - {df['precio'].max():,.0f}]")
print(f"  Precio mediano                  : ${df['precio'].median():,.2f}")
print(f"  Precio promedio                 : ${df['precio'].mean():,.2f}")
print()
print(df[NUM_COLS].describe().round(2))


### 3.2 Propiedades por Provincia

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

por_prov = df["provincia"].value_counts()
por_prov.plot(kind="bar", ax=axes[0], color=COLOR_ACC, edgecolor="white")
axes[0].set_title("Propiedades por Provincia", fontsize=12)
axes[0].set_xlabel("")
axes[0].set_ylabel("Cantidad")
axes[0].tick_params(axis="x", rotation=45)
for p in axes[0].patches:
    axes[0].annotate(f"{p.get_height():,}",
                     (p.get_x() + p.get_width()/2, p.get_height()),
                     ha="center", va="bottom", fontsize=8)

med_prov = df.groupby("provincia")["precio"].median().sort_values(ascending=False)
med_prov.plot(kind="bar", ax=axes[1], color=COLOR_MAIN, edgecolor="white")
axes[1].set_title("Precio mediano por Provincia (USD)", fontsize=12)
axes[1].set_ylabel("Mediana de precio (USD)")
axes[1].tick_params(axis="x", rotation=45)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))

plt.suptitle("Analisis por Provincia", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("figs/por_provincia.png", dpi=150, bbox_inches="tight")
plt.show()
print(por_prov.to_frame("propiedades").join(med_prov.rename("precio_mediano").round(2)))


### 3.3 Propiedades por Lugar

In [ ]:
TOP_N = 20
por_lugar = df["ciudad"].value_counts().head(TOP_N)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(por_lugar.index[::-1], por_lugar.values[::-1],
               color=sns.color_palette("Blues_r", TOP_N))
ax.set_xlabel("Numero de propiedades")
ax.set_title(f"Top {TOP_N} ciudades con mas propiedades", fontsize=13)
for bar, val in zip(bars, por_lugar.values[::-1]):
    ax.text(val + max(por_lugar)*0.01, bar.get_y() + bar.get_height()/2,
            f"{val:,}", va="center", fontsize=9)
plt.tight_layout()
plt.savefig("figs/por_lugar.png", dpi=150, bbox_inches="tight")
plt.show()


### 3.4 Mediana y Promedio de Precio (general y por Ciudad)

In [ ]:
print("ESTADISTICAS DE PRECIO - GENERAL")
print(f"  Mediana : ${df['precio'].median():,.2f}")
print(f"  Promedio: ${df['precio'].mean():,.2f}")
print(f"  Desv. estandar: ${df['precio'].std():,.2f}")
print()

top_ciudades = df["ciudad"].value_counts().head(15).index
stats_lugar = (
    df[df["ciudad"].isin(top_ciudades)]
    .groupby("ciudad")["precio"]
    .agg(["count","median","mean","std"])
    .rename(columns={"count":"n","median":"mediana","mean":"promedio","std":"desv_std"})
    .sort_values("mediana", ascending=False)
)
print("ESTADISTICAS DE PRECIO - TOP 15 CIUDADES")
print(stats_lugar.round(2).to_string())


In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))
top15 = stats_lugar.sort_values("mediana", ascending=True).tail(15)
x     = range(len(top15))
width = 0.4
ax.barh([i - width/2 for i in x], top15["mediana"], height=width,
        label="Mediana", color=COLOR_ACC)
ax.barh([i + width/2 for i in x], top15["promedio"], height=width,
        label="Promedio", color=COLOR_MAIN)
ax.set_yticks(list(x))
ax.set_yticklabels(top15.index, fontsize=9)
ax.set_xlabel("Precio (USD)")
ax.set_title("Mediana vs Promedio de alquiler - Top 15 ciudades", fontsize=13)
ax.legend()
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
plt.tight_layout()
plt.savefig("figs/precio_por_lugar.png", dpi=150, bbox_inches="tight")
plt.show()


### 3.5 Analisis de la relacion entre Area y Precio

In [ ]:
corr = df[["area","precio"]].corr().iloc[0,1]
print(f"Correlacion de Pearson (Area vs Precio): {corr:.4f}")

fig = px.scatter(
    df.sample(min(3000, len(df)), random_state=42),
    x="area", y="precio", color="provincia",
    trendline="ols", opacity=0.6,
    title=f"Area vs Precio de alquiler (r = {corr:.3f})",
    labels={"area":"Area (m2)","precio":"Precio (USD)"},
    template="simple_white", height=480
)
fig.update_traces(marker=dict(size=5))
fig.show()


In [ ]:
df["rango_area"] = pd.cut(df["area"],
    bins=[0,50,80,120,180,250,400,1000],
    labels=["0-50","50-80","80-120","120-180","180-250","250-400","400+"])
area_precio = df.groupby("rango_area", observed=True)["precio"].agg(["median","mean","count"]).round(2)
print("Precio mediano y promedio por rango de area (m2):")
print(area_precio.to_string())

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(area_precio.index.astype(str), area_precio["median"],
       color=COLOR_ACC, edgecolor="white")
ax.set_xlabel("Rango de area (m2)")
ax.set_ylabel("Precio mediano (USD)")
ax.set_title("Precio mediano por rango de area")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
plt.tight_layout()
plt.savefig("figs/area_vs_precio.png", dpi=150, bbox_inches="tight")
plt.show()


### 3.6 Premio por Habitacion Adicional

In [ ]:
price_by_rooms = (
    df[df["num_dormitorios"].between(1, 6)]
    .groupby("num_dormitorios")["precio"]
    .agg(["mean","median","count"])
    .rename(columns={"mean":"promedio","median":"mediana","count":"n"})
)
price_by_rooms.index = price_by_rooms.index.astype(int)
price_by_rooms["premium_abs"] = price_by_rooms["promedio"].diff()
price_by_rooms["premium_pct"] = price_by_rooms["promedio"].pct_change() * 100

print("PREMIO POR HABITACION ADICIONAL (precio promedio)")
print(price_by_rooms.round(2).to_string())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar(price_by_rooms.index, price_by_rooms["promedio"],
            color=sns.color_palette("Blues_r", len(price_by_rooms)))
axes[0].set_xlabel("Numero de dormitorios")
axes[0].set_ylabel("Precio promedio (USD)")
axes[0].set_title("Precio promedio por dormitorios")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))

idx  = price_by_rooms.index[1:]
vals = price_by_rooms["premium_abs"].dropna().values
cols = [COLOR_ACC if v > 0 else COLOR_WARN for v in vals]
axes[1].bar([f"{i-1}->{i}" for i in idx], vals, color=cols, edgecolor="white")
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_xlabel("Transicion de dormitorios")
axes[1].set_ylabel("Delta Precio promedio (USD)")
axes[1].set_title("Premio por habitacion adicional")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))

plt.suptitle("Analisis de Premio por Habitacion", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("figs/premium_habitacion.png", dpi=150, bbox_inches="tight")
plt.show()


### 3.7 Analisis adicionales

In [ ]:
# Distribucion de precio (lineal y logaritmica)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(df["precio"], bins=60, color=COLOR_ACC, edgecolor="white", linewidth=0.4)
axes[0].set_xlabel("Precio (USD)")
axes[0].set_ylabel("Frecuencia")
axes[0].set_title("Distribucion de precio (lineal)")
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))

axes[1].hist(np.log1p(df["precio"]), bins=60, color=COLOR_MAIN, edgecolor="white")
axes[1].set_xlabel("log(1 + Precio)")
axes[1].set_ylabel("Frecuencia")
axes[1].set_title("Distribucion de precio (log)")

plt.suptitle("Distribucion del precio de alquiler", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("figs/distribucion_precio.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Mapa de calor de correlaciones
corr_matrix = df[["precio","num_dormitorios","num_banos","area","num_garages"]].corr()
fig, ax = plt.subplots(figsize=(7, 5))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt=".3f", mask=mask,
            cmap="Blues", ax=ax, linewidths=0.5, vmin=-1, vmax=1)
ax.set_title("Correlacion entre variables numericas", fontsize=12, pad=10)
plt.tight_layout()
plt.savefig("figs/correlacion.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Precio por numero de banos (boxplot)
fig = px.box(
    df[df["num_banos"].between(1, 5)],
    x="num_banos", y="precio", color="num_banos",
    color_discrete_sequence=px.colors.sequential.Blues_r,
    title="Distribucion de precio por numero de banos",
    labels={"num_banos":"Banos","precio":"Precio (USD)"},
    template="simple_white"
)
fig.show()


## 4. Columna Tipo de Precio por Lugar

In [ ]:
# Cuartiles calculados dentro de cada ciudad (lugar canonico)
q_por_lugar = (
    df.groupby("lugar")["precio"]
    .quantile([0.25, 0.75])
    .unstack()
    .rename(columns={0.25: "q1", 0.75: "q3"})
)

df = df.join(q_por_lugar, on="lugar")

def clasificar_precio(row):
    if pd.isna(row["q1"]) or pd.isna(row["q3"]):
        return "Medio"
    if row["precio"] < row["q1"]:
        return "Economico"
    if row["precio"] > row["q3"]:
        return "Lujo"
    return "Medio"

df["tipo_precio"] = df.apply(clasificar_precio, axis=1)
df = df.drop(columns=["q1","q3"])

print("Distribucion de Tipo de Precio por Lugar:")
print(df["tipo_precio"].value_counts())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

counts = df["tipo_precio"].value_counts()
axes[0].pie(counts, labels=counts.index, autopct="%1.1f%%",
            colors=["#2e86c1","#85c1e9","#1a5276"],
            startangle=140, textprops={"fontsize":11})
axes[0].set_title("Distribucion global de Tipo de Precio")

pivot     = df.groupby(["provincia","tipo_precio"]).size().unstack(fill_value=0)
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
cats = [c for c in ["Economico","Medio","Lujo"] if c in pivot_pct.columns]
pivot_pct[cats].plot(kind="bar", stacked=True, ax=axes[1],
    color=["#85c1e9","#2e86c1","#1a5276"], edgecolor="white")
axes[1].set_title("Tipo de Precio por Provincia (%)")
axes[1].set_ylabel("Porcentaje (%)")
axes[1].tick_params(axis="x", rotation=45)
axes[1].legend(title="Tipo de Precio")

plt.suptitle("Clasificacion de Precio por Lugar", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("figs/tipo_precio.png", dpi=150, bbox_inches="tight")
plt.show()


## 5. Exportacion del Dataset Limpio

In [ ]:
os.makedirs("data", exist_ok=True)
CLEAN_PATH = "data/alquileres_clean.csv"
df.to_csv(CLEAN_PATH, index=False, encoding="utf-8")
print(f"Dataset limpio guardado en: {CLEAN_PATH}")
print(f"Shape final: {df.shape}")
print("\nColumnas disponibles:")
for c in df.columns:
    print(f"  {c}: {df[c].dtype}")


---
## Resumen del EDA

| Aspecto | Descripcion |
|---|---|
| Encoding | Carga robusta con reparacion de artefactos latin-1/UTF-8 |
| Lugar | Parser jerarquico: Provincia > Sector > Ciudad > Ecuador |
| Imputacion | Mediana contextual por provincia+ciudad |
| Outliers | Filtrado por percentil 99.5 y precio minimo $10 |
| Correlacion area-precio | Positiva moderada (ver heatmap) |
| Premio por habitacion | Mayor salto en transiciones 1->2 y 2->3 dormitorios |
| Tipo de Precio | Basado en Q1/Q3 dentro de cada ciudad |

**Siguiente paso:** `02_modelado_ml.ipynb`
